# install packages

In [1]:
!pip install gensim
!pip install parsivar
!pip install torch==2.2.2 torchtext==0.17.2

  Using cached torch-2.2.2-cp312-cp312-manylinux1_x86_64.whl.metadata (25 kB)
  Using cached torchtext-0.17.2-cp312-cp312-manylinux1_x86_64.whl.metadata (7.9 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl.metad

# import packages

In [2]:
from torch.utils.data import Dataset
from transformers import MBartTokenizer
from torchtext.vocab import build_vocab_from_iterator
from datasets import load_dataset
from typing import Iterable, List
import torch.nn as nn
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from tqdm import tqdm
import random
from torch.nn.utils.rnn import pad_sequence
import torch
import gensim
import torch
import numpy as np
from torch.utils.data import DataLoader
import torch
from sklearn.model_selection import train_test_split
from gensim.models import Word2Vec
from parsivar import Tokenizer
import sentencepiece as spm
import math
from collections import Counter

# Neural machine Translation

In [3]:
dataset = load_dataset("shenasa/English-Persian-Parallel-Dataset", split='train')

print(dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset.tsv:   0%|          | 0.00/872M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3960172 [00:00<?, ? examples/s]

{'flash fire .': 'superheats the air . burns the lungs like rice paper .', 'فلاش آتش .': 'هوا را فوق العاده گرم می کند . ریه ها را مثل کاغذ برنج می سوزاند .'}


In [4]:
dataset

Dataset({
    features: ['flash fire .', 'فلاش آتش .'],
    num_rows: 3960172
})

## training tokenizer and pretraining word2vec


In [5]:
sample_size_1 = 160000
sampled_dataset_1 = dataset.shuffle(seed=42).select(range(sample_size_1))

# Extract English and Persian sentences
english_sentences_1 = [item['flash fire .'] for item in sampled_dataset_1]
persian_sentences_1 = [item['فلاش آتش .'] for item in sampled_dataset_1]

In [6]:
# Save sentences to a text file (BPE requires input in file format)
with open("persian_corpus.txt", "w", encoding="utf-8") as f:
    for sentence in persian_sentences_1:
        f.write(sentence + "\n")

In [7]:
# Path to the corpus file
corpus_file = "/content/persian_corpus.txt"

# Set parameters for training the tokenizer
vocab_size = 21000  # Adjust the vocabulary size
character_coverage = 0.999 # Coverage of characters in the training data
model_prefix = "/content/persian_spm"  # Prefix for saving the model and vocab

# Train the SentencePiece model
spm.SentencePieceTrainer.train(
    input=corpus_file,
    model_prefix=model_prefix,
    vocab_size=vocab_size,
    character_coverage=character_coverage,
    model_type='bpe'
)


In [8]:
# Tokenized Persian sentences (list of lists of tokens)

tokenizer = Tokenizer()
tokenized_sentences_1 = [tokenizer.tokenize_words(sentence) for sentence in persian_sentences_1]

vector_size=300
window=5
min_count=3
workers=4
# Train Word2Vec
word2vec_model = Word2Vec(
    sentences=tokenized_sentences_1,
    vector_size=vector_size,
    window=window,
    min_count=min_count,
    workers=workers
)

# Save the Word2Vec model
word2vec_model.save("word2vec_persian.model")

In [9]:
mb = MBartTokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# Tokenized Persian sentences (list of lists of tokens)
tokenized_sentences_1 = [mb.tokenize(sentence) for sentence in english_sentences_1]
vector_size=300
window=5
min_count=3
workers=4

# Train Word2Vec
word2vec_model = Word2Vec(
    sentences=tokenized_sentences_1,
    vector_size=vector_size,
    window=window,
    min_count=min_count,
    workers=workers
)
# Save the Word2Vec model
word2vec_model.save("word2vec_english.model")

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.


## Download dataset

In [10]:
# Randomly sample 20,000 examples from the dataset
sample_size = 10000
sampled_dataset = dataset.shuffle(seed=42).select(range(sample_size))

# Extract English and Persian sentences
english_sentences = [item['flash fire .'] for item in sampled_dataset]
persian_sentences = [item['فلاش آتش .'] for item in sampled_dataset]

print(f"Number of samples in the subset: {len(english_sentences)}")
print("First English sentence:", english_sentences[10])
print("First Persian sentence:", persian_sentences[10])

Number of samples in the subset: 10000
First English sentence: his wings have been cut off .
First Persian sentence: بالهایش بریده شده است .


## Train valid Test split & Creating Datasets

In [11]:
# Define the proportion for the train-test split
test_size = 0.10  # 20% of data for testing
random_seed = 42  # For reproducibility

# Split the data into train and test sets
english_train, english_test, persian_train, persian_test = train_test_split(
    english_sentences, persian_sentences, test_size=test_size, random_state=random_seed
)

english_test, english_valid, persian_test, persian_valid = train_test_split(
    english_test, persian_test, test_size=0.4, random_state=random_seed
)

# Display the size of each split
print(f"Number of training samples: {len(english_train)}")
print(f"Number of validation samples: {len(english_valid)}")
print(f"Number of testing samples: {len(english_test)}")

Number of training samples: 9000
Number of validation samples: 400
Number of testing samples: 600


In [12]:
# Custom Dataset class for English-Persian translation
class TranslationDataset(Dataset):
    def __init__(self, src_data, tgt_data):
        self.src_data = src_data
        self.tgt_data = tgt_data

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]
# Create Dataset
train_dataset = TranslationDataset(english_train, persian_train)
valid_dataset = TranslationDataset(english_valid, persian_valid)
test_dataset = TranslationDataset(english_test, persian_test)

In [13]:
print(english_train)

['Semmler was grinning. I told you.', '"""When the seven loaves fed the four thousand, how many baskets full of broken pieces did you take up?"" They told him, ""Seven."""', 'What the fuck is this?', "Mikhail Shmakov, left, chairman of the Russian Federation of Independent Trade Unions, and federation's secretary Aleksei Zharkov at the Labour 20 Summit.", "This product hasn't been delivered to the Russian market by the official channel since 2007.", 'With our team running like a well-oiled machine, we can tackle any project at the drop of a hat, ensuring we always keep our noses to the grindstone.', 'on the contrary, Jurgis scrubbed the spittoons and polished the banisters all the more vehemently because at the same time he was wrestling inwardly with an imaginary recalcitrant.', 'EYEleds.pl', 'Loaded at: Sep 11, 2010, 1:27:35 AM (7 years ago)', 'Transmitted: 12/9/2017 6:06:18 AM', '"It is awful!"" 62126"', 'Second place: DJ Flip (Ireland) (http://www.mixcloud.com/RedBullThre3style/dj-

## Build vocabulary

In [14]:
token_transform = {}
mb = MBartTokenizer.from_pretrained("facebook/mbart-large-50")  # Initialize MBartTokenizer
def en_tokenize(sentence):
    return mb.tokenize(sentence)

token_transform['en'] = en_tokenize

token_transform = {}
mb = MBartTokenizer.from_pretrained("facebook/mbart-large-50")  # Initialize MBartTokenizer
def en_tokenize(sentence):
    return mb.tokenize(sentence)

token_transform['en'] = en_tokenize


sp = spm.SentencePieceProcessor()
sp.load('/content/persian_spm.model')
# Define the tokenization function using SentencePiece
def persian_tokenize(sentence):
    return sp.tokenize(sentence, out_type=str)
token_transform['fa'] = persian_tokenize
# تعریف نمادهای ویژه
UNK_IDX, PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2, 3
special_symbols = ['<unk>', '<pad>', '<bos>', '<eos>']

# دیکشنری برای ذخیره واژگان هر زبان
vocab_transform = {}

# تابعی برای ایجاد توکن‌ها از داده‌ها
# Function to yield individual tokens (one by one)
def yield_tokens(data_iter, language: str):
    language_index = {'en': 0, 'fa': 1}
    for i,data_sample in enumerate(data_iter):

        yield token_transform[language](data_sample[language_index[language]])

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'MBart50Tokenizer'. 
The class this function is called from is 'MBartTokenizer'.


In [15]:
for ln in ['fa', 'en']:
    print(f"Building vocabulary for {ln}...")
    # Create training iterator
    train_iterator = list(zip(english_train, persian_train))
    sorted_dataset = sorted(train_iterator, key=lambda x: len(x[0].split()))  # Sort by sentence length

    # Build vocabulary
    vocab_transform[ln] = build_vocab_from_iterator(
        yield_tokens(sorted_dataset, ln),
        specials=special_symbols,
        special_first=True
    )

# Set default index for both vocabularies
vocab_transform['en'].set_default_index(UNK_IDX)
vocab_transform['fa'].set_default_index(UNK_IDX)

en_vocab_size = len(vocab_transform['en'])
fa_vocab_size = len(vocab_transform['fa'])
# Print vocab size for verification
print("English vocab size:",en_vocab_size )
print("Persian vocab size:",fa_vocab_size)

print(list(vocab_transform['fa'].get_stoi().keys())[-10:])
print(list(vocab_transform['en'].get_stoi().keys())[:30])

Building vocabulary for fa...
Building vocabulary for en...
English vocab size: 15235
Persian vocab size: 13827
['61', '▁غلط', 'لومین', '▁نشد', '▁مو', 'لیس', '▁عملکردها', '▁KazMunayGas', '▁ادبیات', 'لیل']
['▁zoom', '▁zo', '▁zij', '▁yurt', '▁xem', '▁widget', '▁whilst', '▁weld', '▁wax', '▁wad', '▁voucher', '▁von', '▁volatil', '▁visibili', '▁vir', '▁vip', '▁vendor', '▁vehement', '▁vasta', '▁vampir', '▁uten', '▁uro', '▁upgrade', '▁unsur', '▁ulterior', '▁ukr', '▁ube', '▁twa', '▁tunnel', '▁tubercul']


In [16]:
print(vocab_transform['en'](['I','▁like','▁machine','▁learning']))
sentence1= "من عاشق کتابها هستم"
sentence2= "من"
print(token_transform['fa'](sentence1))
print(vocab_transform['fa'](token_transform['fa'](sentence1)))
print(vocab_transform['fa'](token_transform['fa'](sentence2)))

[310, 119, 1180, 3071]
['▁من', '▁عاشق', '▁کتاب', 'ها', '▁هستم']
[31, 3546, 632, 343, 481]
[31]


## Prepare data to input model
we make  train_dataloader and  valid_dataloader in this step.

### tensor transforms

In [17]:
# Tokenization and vocabulary for both languages
SRC_LANGUAGE = 'en'
TGT_LANGUAGE = 'fa'

# Set device to GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Transform functions for source and target tokens
def tensor_transform_s(token_ids: List[int]):
    return torch.cat((
        torch.tensor([BOS_IDX]),
        torch.tensor(token_ids[::-1]),
        torch.tensor([EOS_IDX])
    ))

def tensor_transform_t(token_ids: List[int]):
    return torch.cat((
        torch.tensor([BOS_IDX]),      # Add beginning-of-sequence token
        torch.tensor(token_ids),      # Keep token order as is (target sentences)
        torch.tensor([EOS_IDX])       # Add end-of-sequence token
    ))
def sequential_transforms(*transforms):
    def func(txt_input):
        for transform in transforms:
            txt_input = transform(txt_input)
        return txt_input
    return func

### Define text transformations

In [18]:
flip = True
text_transform = {
    SRC_LANGUAGE: sequential_transforms(
        lambda x: vocab_transform[SRC_LANGUAGE](token_transform[SRC_LANGUAGE](x)),
        tensor_transform_s
    ),
    TGT_LANGUAGE: sequential_transforms(
        lambda x: vocab_transform[TGT_LANGUAGE](token_transform[TGT_LANGUAGE](x)),
        tensor_transform_t
    )
}


### collate_fn

In [19]:
# Collate function to pad and batch the data
def collate_fn(batch):
    src_batch, tgt_batch = [], []

    for src_sample, tgt_sample in batch:
        src_batch.append(text_transform[SRC_LANGUAGE](src_sample.rstrip("\n")))
        tgt_batch.append(text_transform[TGT_LANGUAGE](tgt_sample.rstrip("\n")))

    # Pad sequences to the same length
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX, batch_first=True)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX, batch_first=True)

    print(f"src_batch in collate_fn: {src_batch.size()}")
    print(f"tgt_batch in collate_fn: {tgt_batch.size()}")

    return src_batch.to(device), tgt_batch.to(device)



In [20]:
# Function to create DataLoader
def get_translation_dataloaders(batch_size=64):

    # Sort by length of the source sentence for efficiency
    sorted_traindataset = sorted(train_dataset, key=lambda x: len(x[0].split()))
    sorted_validdataset = sorted(valid_dataset, key=lambda x: len(x[0].split()))
    sorted_testdataset = sorted(test_dataset, key=lambda x: len(x[0].split()))

    # Create DataLoader
    train_dataloader = DataLoader(
        sorted_traindataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        drop_last=True
    )
    valid_dataloader = DataLoader(
        sorted_validdataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        drop_last=True
    )
    test_dataloader = DataLoader(
        sorted_testdataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        drop_last=True
    )

    return train_dataloader,valid_dataloader,test_dataloader
train_dataloader,valid_dataloader,test_dataloader = get_translation_dataloaders(batch_size=4)

## Translation model

### Load Word2Vec model for persian

In [21]:
word2vec_model = gensim.models.Word2Vec.load('/content/word2vec_persian.model')

# Get vocabulary size and embedding dimension
vocab_size = fa_vocab_size
embedding_dim = word2vec_model.vector_size

# Initialize embedding matrix with random values
embedding_matrix = np.random.uniform(-0.1, 0.1, (vocab_size, embedding_dim))

# Get the vocabulary from your vocab_transform dictionary
vocab = vocab_transform['fa'].get_itos() # Assuming 'fa' is the key for your Persian vocabulary

# Fill embedding matrix with Word2Vec vectors for words in the vocabulary
for i, word in enumerate(vocab):
    if word in word2vec_model.wv:
        embedding_matrix[i] = word2vec_model.wv[word]

# Convert to torch tensor
embedding_matrix_fa = torch.tensor(embedding_matrix, dtype=torch.float).to(device)
print(embedding_matrix_fa.shape)


torch.Size([13827, 300])


### Load Word2Vec model for english

In [22]:
# Load Word2Vec model for English
word2vec_model_en = gensim.models.Word2Vec.load('/content/word2vec_english.model')

# Get vocabulary size and embedding dimension for English
vocab_size_en = en_vocab_size
embedding_dim_en = word2vec_model_en.vector_size

# Initialize embedding matrix with random values for English
embedding_matrix_en = np.random.uniform(-0.1, 0.1, (vocab_size_en, embedding_dim_en))

# Get the vocabulary from your vocab_transform dictionary for English
vocab_en = vocab_transform['en'].get_itos()

# Fill embedding matrix with Word2Vec vectors for words in the English vocabulary
for i, word in enumerate(vocab_en):
    if word in word2vec_model_en.wv:
        embedding_matrix_en[i] = word2vec_model_en.wv[word]

# Convert to torch tensor
embedding_matrix_en = torch.tensor(embedding_matrix_en, dtype=torch.float).to(device)
print(embedding_matrix_en.shape)

torch.Size([15235, 300])


In [23]:
class Encoder(nn.Module):
    def __init__(self, vocab_len, emb_dim, hid_dim, n_layers, dropout_prob):
        super().__init__()

        # Define the embedding layer
        self.embedding = nn.Embedding(vocab_len, emb_dim)

        # Define the LSTM layer
        self.rnn = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hid_dim,
            num_layers=n_layers,
            dropout=dropout_prob,
            batch_first=True
        )

        # Store n_layers, hid_dim, and device as instance attributes
        self.num_layers = n_layers
        self.hidden_size = hid_dim
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)
        cell = torch.zeros(self.num_layers, batch_size, self.hidden_size).to(self.device)
        return hidden, cell

    def forward(self, input_batch):
        # Get word embeddings for the input batch
        embedded = self.embedding(input_batch)  # shape: (batch_size, seq_len, emb_dim)

        # Pass through the LSTM
        output, (hidden, cell) = self.rnn(embedded)

        return hidden, cell

In [24]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()

        self.output_dim = output_dim

        # Define the embedding layer
        self.embedding = nn.Embedding(output_dim, emb_dim)

        # Define the LSTM layer
        self.rnn = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hid_dim,
            num_layers=n_layers,
            dropout=dropout,
            batch_first=True
        )

        # Define a fully connected layer to output predictions
        self.fc_out = nn.Linear(hid_dim, output_dim)

        # Define a dropout layer
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell):
        # Pass the input token through the embedding layer
        embedded = self.embedding(input)  # shape: (batch_size, emb_dim)

        # Apply dropout on embedded input
        embedded = self.dropout(embedded)

        # Pass the embedded input and previous hidden, cell states through the LSTM
        output, (hidden, cell) = self.rnn(embedded.unsqueeze(1), (hidden, cell))  # unsqueeze to match LSTM input shape

        # Convert the hidden state to the output space using a fully connected layer
        prediction = self.fc_out(output.squeeze(1))  # shape: (batch_size, output_dim)

        return prediction, hidden, cell


In [25]:
class Seq2Seq(nn.Module):
    def __init__(self,encoder, decoder, device, trg_vocab):
      super().__init__()
      self.encoder = encoder
      self.decoder = decoder
      self.device = device
      self.trg_vocab = trg_vocab
      self.trg_vocab_size = len(trg_vocab) # Add this line to store the vocabulary size


    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = trg.size(0)  # Batch-first ensures batch_size comes first
        trg_len = trg.size(1)    # trg shape is (batch_size, trg_seq_len)
        trg_vocab_size = self.decoder.output_dim  # Vocabulary size of the decoder

        # Tensor to store decoder outputs
        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        # Pass the source sequence through the encoder
        hidden, cell = self.encoder(src)

        # Initial input to the decoder is the <sos> token for all batches
        input = trg[:, 0]

        for t in range(1, trg_len):
            # Pass the input, hidden, and cell state to the decoder
            output, hidden, cell = self.decoder(input, hidden, cell)

            # Store the output
            outputs[:, t, :] = output

            # Decide whether to use teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio

            # Get the highest probability prediction
            top1 = output.argmax(1)

            # Decide the next input: ground truth or prediction
            input = trg[:, t] if teacher_force else top1

        return outputs

In [26]:
emb_dim = 300
hid_dim = 300
n_layers = 3
dropout_prob= 0.2
encoder = Encoder(en_vocab_size, emb_dim, hid_dim, n_layers, dropout_prob)
output_dim = fa_vocab_size
decoder = Decoder(output_dim, emb_dim, hid_dim, n_layers, dropout_prob)
trg_vocab = vocab_transform['fa']
model = Seq2Seq(encoder, decoder, device, trg_vocab).to(device)



## Training and Validation

In [27]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

PAD_IDX = 1  # Change this according to your setup

#criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

def compute_loss(outputs, trg):
    """
    Compute the loss for the Seq2Seq model.

    Args:
    outputs: Predicted logits from the model, shape (trg_len, batch_size, trg_vocab_size)
    trg: Ground truth target sequence, shape (trg_len, batch_size)
    trg_vocab_size: Size of the target vocabulary

    Returns:
    loss: Computed loss value
    """
    # Flatten outputs and targets
    outputs = outputs.view(-1, outputs.shape[-1])  # (batch_size * trg_len, trg_vocab_size)
    print("Output shape:", outputs.shape)
    trg = trg.view(-1)  # (batch_size * trg_len)
    print("Target shape:", trg.shape)
    return criterion(outputs, trg)

In [28]:
def index_to_eng(seq_en):
    return " ".join([vocab_transform['en'].get_itos()[index.item()] for index in seq_en])

def index_to_german(seq_de):
    return " ".join([vocab_transform['de'].get_itos()[index.item()] for index in seq_de])

In [29]:

def train(model, iterator, optimizer, clip):
    model.train()
    epoch_loss = 0
    train_iterator = tqdm(iterator, desc="Training", leave=False)

    for i, (src, trg) in enumerate(train_iterator):
        # Move data to device (e.g., GPU if available)
        src, trg = src.to(model.device), trg.to(model.device)
        print(f"src : {src.size()}")
        print(f"trg : {trg.size()}")
        # Initialize hidden and cell states with correct batch size
        batch_size = src.size(0)
        hidden, cell = model.encoder.init_hidden(batch_size)
        print("Input batch size:", src.size(0))
        print("Hidden state shape:", hidden.size())
        print("Cell state shape:", cell.size())
        # Output prediction from the model
        output = model(src, trg, teacher_forcing_ratio=0.5)

        # Adjust dimensions for computing loss
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)  # Flatten (trg_len-1) * batch_size * trg_vocab_size
        print("Output shape:", output.shape)
        trg = trg[1:].view(-1)  # Flatten (trg_len-1) * batch_size
        print("Target shape:", trg.shape)


        # Compute the loss
        loss = compute_loss(output, trg)
        loss.backward()

        # Clip gradients to avoid exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        # Step the optimizer
        optimizer.step()

        # Track the loss
        train_iterator.set_postfix(loss=loss.item())
        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [30]:

def evaluate(model, iterator):
    model.eval()
    epoch_loss = 0
    valid_iterator = tqdm(iterator, desc="Evaluating", leave=False)

    with torch.no_grad():
        for i, (src, trg) in enumerate(valid_iterator):
            # Move data to device
            src, trg = src.to(model.device), trg.to(model.device)

            # Initialize hidden and cell states with correct batch size
            batch_size = src.size(0)  # Changed from src.size(1) to src.size(0)
            hidden, cell = model.encoder.init_hidden(batch_size)
            print("Input batch size:", src.size(0))
            print("Hidden state shape:", hidden.size())
            print("Cell state shape:", cell.size())
            # Output prediction from the model
            output = model(src, trg, teacher_forcing_ratio=0)  # No teacher forcing in evaluation

            # Adjust dimensions for computing loss
            output_dim = output.shape[0]
            output = output[1:].view(-1, output_dim)  # Flatten (trg_len-1) * batch_size * trg_vocab_size
            trg = trg[1:].view(-1)  # Flatten (trg_len-1) * batch_size

            # Compute the loss
            loss = compute_loss(output, trg)

            epoch_loss += loss.item()

    return epoch_loss / len(iterator)


In [31]:
def generate_translation(model, src_sentence, src_vocab, trg_vocab, max_len=50):
    model.eval()
    with torch.no_grad():
        # Convert src_sentence to tensor
        src_tensor = torch.tensor([src_vocab.get_stoi().get(word, UNK_IDX) for word in src_sentence.split()]).unsqueeze(0).to(model.device)

        # Pass the source tensor through the encoder
        hidden, cell = model.encoder(src_tensor)

        # Initialize the target sequence with the <bos> token
        trg_indexes = [trg_vocab.get_stoi()['<bos>']]
        trg_tensor = torch.tensor([trg_indexes[0]]).to(model.device)  # Remove extra unsqueeze

        batch_size = src_tensor.size(0)

        hidden = hidden.reshape(model.decoder.rnn.num_layers, batch_size, model.decoder.rnn.hidden_size)
        cell = cell.reshape(model.decoder.rnn.num_layers, batch_size, model.decoder.rnn.hidden_size)

        for i in range(max_len):
            # Output prediction from the decoder
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

            # Get the most probable token (argmax)
            pred_token = output.argmax(1).item()  # .item() added to get scalar value
            trg_indexes.append(pred_token)

            # Stop if the <eos> token is generated
            if pred_token == trg_vocab.get_stoi()['<eos>']:
                break

            # Prepare the next input for the decoder
            trg_tensor = torch.tensor([pred_token]).to(model.device)  # Remove extra unsqueeze

        # Convert token indexes to words
        trg_tokens = [trg_vocab.get_itos()[i] for i in trg_indexes]

        # Remove <bos> and <eos> tokens
        trg_tokens = trg_tokens[1:]

        return " ".join(trg_tokens)

In [32]:
# Define the optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
train_iterator = train_dataloader
valid_iterator = valid_dataloader
clip = 1
epochs = 3
tlosses = []
vlosses = []

# Example source sentences for translation
src_sentences = ["I want to go home.","you should translate this sentence.","how are you?","do you want to eat cake?"]
src_vocab = vocab_transform['en']
trg_vocab = vocab_transform['fa']

for epoch in range(epochs):
    # Train the model for one epoch
    translation = generate_translation(model, src_sentences[epoch % 4], src_vocab, trg_vocab)
    print(f"Translation: {translation}")
    train_loss = train(model, train_iterator, optimizer, clip)
    tlosses.append(train_loss)
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Train Loss: {train_loss:.4f}")

    # Generate a translation for a sample sentence
    print('-' * 40)

Translation: ▁طغیان ▁طغیان ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35 ▁35


Training:   0%|          | 0/2250 [00:00<?, ?it/s]

src_batch in collate_fn: torch.Size([4, 22])
tgt_batch in collate_fn: torch.Size([4, 22])
src : torch.Size([4, 22])
trg : torch.Size([4, 22])
Input batch size: 4
Hidden state shape: torch.Size([3, 4, 300])
Cell state shape: torch.Size([3, 4, 300])
Output shape: torch.Size([66, 13827])
Target shape: torch.Size([66])
Output shape: torch.Size([66, 13827])
Target shape: torch.Size([66])


Training:   0%|          | 2/2250 [00:00<16:29,  2.27it/s, loss=9.54]

src_batch in collate_fn: torch.Size([4, 55])
tgt_batch in collate_fn: torch.Size([4, 68])
src : torch.Size([4, 55])
trg : torch.Size([4, 68])
Input batch size: 4
Hidden state shape: torch.Size([3, 4, 300])
Cell state shape: torch.Size([3, 4, 300])
Output shape: torch.Size([204, 13827])
Target shape: torch.Size([204])
Output shape: torch.Size([204, 13827])
Target shape: torch.Size([204])
src_batch in collate_fn: torch.Size([4, 7])
tgt_batch in collate_fn: torch.Size([4, 10])
src : torch.Size([4, 7])
trg : torch.Size([4, 10])
Input batch size: 4
Hidden state shape: torch.Size([3, 4, 300])
Cell state shape: torch.Size([3, 4, 300])
Output shape: torch.Size([30, 13827])
Target shape: torch.Size([30])
Output shape: torch.Size([30, 13827])
Target shape: torch.Size([30])


...

Training: 100%|█████████▉| 2248/2250 [02:48<00:00,  2.99it/s, loss=6.78]

src_batch in collate_fn: torch.Size([4, 172])
tgt_batch in collate_fn: torch.Size([4, 122])
src : torch.Size([4, 172])
trg : torch.Size([4, 122])
Input batch size: 4
Hidden state shape: torch.Size([3, 4, 300])
Cell state shape: torch.Size([3, 4, 300])
Output shape: torch.Size([366, 13827])
Target shape: torch.Size([366])
Output shape: torch.Size([366, 13827])
Target shape: torch.Size([366])


Training: 100%|█████████▉| 2249/2250 [02:48<00:00,  3.07it/s, loss=6.82]

src_batch in collate_fn: torch.Size([4, 219])
tgt_batch in collate_fn: torch.Size([4, 187])
src : torch.Size([4, 219])
trg : torch.Size([4, 187])
Input batch size: 4
Hidden state shape: torch.Size([3, 4, 300])
Cell state shape: torch.Size([3, 4, 300])
Output shape: torch.Size([561, 13827])
Target shape: torch.Size([561])
Output shape: torch.Size([561, 13827])
Target shape: torch.Size([561])


Epoch 3/3
Train Loss: 6.3683
----------------------------------------


In [33]:
nltk.download('punkt')  # For tokenization if needed

# If nltk not available, use this from-scratch BLEU (simplified for 1-4 grams)
def compute_bleu(references, candidates, max_n=4, weights=(0.25, 0.25, 0.25, 0.25)):

    def ngram_precision(ref, cand, n):
        cand_ngrams = Counter([tuple(cand[i:i+n]) for i in range(len(cand)-n+1)])
        ref_ngrams = Counter([tuple(ref[i:i+n]) for i in range(len(ref)-n+1)])
        matches = sum(min(cand_ngrams[ng], ref_ngrams[ng]) for ng in cand_ngrams)
        total = sum(cand_ngrams.values())
        return matches / total if total > 0 else 0

    def brevity_penalty(ref_len, cand_len):
        if cand_len > ref_len:
            return 1
        return math.exp(1 - ref_len / cand_len) if cand_len > 0 else 0

    scores = []
    for ref, cand in zip(references, candidates):
        ref = nltk.word_tokenize(ref)  # Tokenize if not already
        cand = nltk.word_tokenize(cand)
        ref_len, cand_len = len(ref), len(cand)
        precisions = [ngram_precision(ref, cand, n) for n in range(1, max_n+1)]
        geo_mean = math.exp(sum(w * math.log(p + 1e-6) for w, p in zip(weights, precisions)))
        bp = brevity_penalty(ref_len, cand_len)
        scores.append(bp * geo_mean)
    return sum(scores) / len(scores) * 100  # Scale to 0-100

# If torchtext vocab is broken, manual vocab builder as fallback
def build_vocab(sentences, specials=['<unk>', '<pad>', '<bos>', '<eos>']):
    vocab = {special: idx for idx, special in enumerate(specials)}
    for sent in sentences:
        for token in sent.split():  # Simple split; replace with your tokenizer
            if token not in vocab:
                vocab[token] = len(vocab)
    return vocab

# If needed: src_vocab = build_vocab(english_sentences, specials)
# trg_vocab = build_vocab(persian_sentences, specials)

# Test function
def evaluate_model(model, test_iterator, src_vocab, trg_vocab):
    model.eval()  # Set to evaluation mode
    candidates = []
    references = []  # List of lists for BLEU (each ref is a list of tokens)

    with torch.no_grad():
        for src_batch, tgt_batch in test_iterator:
            for i in range(len(src_batch)):
                src_sentence = src_batch[i]  # Assume pre-tokenized or convert back to string
                ref = tgt_batch[i]  # Reference Persian sentence
                # Convert tensor to string if needed (depends on your preprocessing)
                src_str = ' '.join([src_vocab.get_itos()[tok.item()] for tok in src_sentence if tok.item() != src_vocab['<pad>']])
                ref_str = ' '.join([trg_vocab.get_itos()[tok.item()] for tok in ref if tok.item() != trg_vocab['<pad>']])

                # Generate candidate translation
                candidate = generate_translation(model, src_str, src_vocab, trg_vocab)

                candidates.append(candidate)
                references.append([ref_str])  # BLEU expects list of references per candidate

    # Compute BLEU
    smoothie = SmoothingFunction().method4  # Handle zero counts
    bleu = corpus_bleu(references, candidates, smoothing_function=smoothie)
    print(f"Test BLEU Score: {bleu * 100:.2f}")

    return bleu

# Create test_dataloader if not already (batch_size=4 like training)
test_dataloader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)  # Assume collate_fn defined

# Run evaluation
bleu_score = evaluate_model(model, test_dataloader, src_vocab, trg_vocab)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


src_batch in collate_fn: torch.Size([4, 40])
tgt_batch in collate_fn: torch.Size([4, 40])
src_batch in collate_fn: torch.Size([4, 26])
tgt_batch in collate_fn: torch.Size([4, 25])
src_batch in collate_fn: torch.Size([4, 28])
tgt_batch in collate_fn: torch.Size([4, 23])
src_batch in collate_fn: torch.Size([4, 29])
tgt_batch in collate_fn: torch.Size([4, 23])
src_batch in collate_fn: torch.Size([4, 37])
tgt_batch in collate_fn: torch.Size([4, 31])
src_batch in collate_fn: torch.Size([4, 52])
tgt_batch in collate_fn: torch.Size([4, 48])
src_batch in collate_fn: torch.Size([4, 66])
tgt_batch in collate_fn: torch.Size([4, 52])
src_batch in collate_fn: torch.Size([4, 87])
tgt_batch in collate_fn: torch.Size([4, 74])
src_batch in collate_fn: torch.Size([4, 32])
tgt_batch in collate_fn: torch.Size([4, 25])
src_batch in collate_fn: torch.Size([4, 60])
tgt_batch in collate_fn: torch.Size([4, 54])
src_batch in collate_fn: torch.Size([4, 38])
tgt_batch in collate_fn: torch.Size([4, 23])
src_batch 